# **Indexing: Inverted Index**


In [46]:
!pip install python-terrier

In [47]:
import pandas as pd
import pyterrier as pt
import os
import re

In [48]:
if not pt.java.started():
    pt.java.init()

In [49]:
df=pd.read_csv('/content/egyptian_movies_preprocessed.csv')
df.head()

,doc_id,title,type,year,rating,original_abstract,processed_text
0,1,The Venal Gentlemen,movie,1983.0,6.000,"Mahmoud, a simple employee of the Ministry of ...",venal gentlemen mahmoud simpl employe ministri...
1,2,The Passage,movie,2019.0,7.042,"A platoon of Commandos’ soldiers, lead by a fe...",passag platoon commando soldier lead fearless ...
2,3,Karmouz War,movie,2018.0,6.000,"Alexandria, Egypt, 1940. Three young Egyptians...",karmouz war alexandria egypt 1940 three young ...
3,4,The Emigrant,movie,1994.0,5.684,The biblical tale of Joseph is told from an Eg...,emigr biblic tale joseph told egyptian perspec...
4,5,The Swarm,movie,2024.0,4.200,Based on actual events and the efforts of the ...,swarm base actual event effort egyptian nation...


In [50]:
df['docno']=df['doc_id'].astype(str)
df[['docno', 'processed_text']].head()

,docno,processed_text
0,1,venal gentlemen mahmoud simpl employe ministri...
1,2,passag platoon commando soldier lead fearless ...
2,3,karmouz war alexandria egypt 1940 three young ...
3,4,emigr biblic tale joseph told egyptian perspec...
4,5,swarm base actual event effort egyptian nation...


In [51]:
index_path=os.path.abspath("egyptianMoviesIndex")
os.makedirs(index_path, exist_ok=True)
indexer=pt.DFIndexer(index_path, overwrite=True)
index_ref=indexer.index(df["processed_text"], df["docno"])
index=pt.IndexFactory.of(index_ref)

/tmp/ipykernel_3878/3217514915.py:3: DeprecationWarning: Call to deprecated class DFIndexer. (use pt.terrier.IterDictIndexer().index(dataframe.to_dict(orient='records')) instead) -- Deprecated since version 0.11.0.
  indexer=pt.DFIndexer(index_path, overwrite=True)


In [52]:
print(index_ref.toString())

/content/egyptianMoviesIndex/data.properties


In [53]:
print(index.getCollectionStatistics().toString())

Number of documents: 5036
Number of terms: 11377
Number of postings: 120404
Number of fields: 0
Number of tokens: 136200
Field names: []
Positions:   false



In [54]:
inverted_index={}
for term_entry in index.getLexicon():
    term=term_entry.getKey()

    if term.isdigit() or re.match(r'^\d+\w*$', term):
        continue

    pointer=term_entry.getValue()

    doc_list=[]
    for posting in index.getInvertedIndex().getPostings(pointer):
        doc_id=posting.getId()
        frequency=posting.getFrequency()
        doc_list.append({'doc_id': doc_id, 'frequency': frequency})

    inverted_index[term]=doc_list

In [55]:
sample_terms=list(inverted_index.keys())[:10]
for term in sample_terms:
    print(f"Term: '{term}' -> {inverted_index[term]}")

Term: 'aa' -> [{'doc_id': 142, 'frequency': 1}, {'doc_id': 1181, 'frequency': 1}, {'doc_id': 2358, 'frequency': 1}, {'doc_id': 4448, 'frequency': 1}]
Term: 'aaan' -> [{'doc_id': 561, 'frequency': 1}]
Term: 'aad' -> [{'doc_id': 212, 'frequency': 1}, {'doc_id': 439, 'frequency': 1}, {'doc_id': 1268, 'frequency': 1}, {'doc_id': 1637, 'frequency': 1}, {'doc_id': 3202, 'frequency': 1}]
Term: 'aal' -> [{'doc_id': 559, 'frequency': 1}, {'doc_id': 2058, 'frequency': 2}, {'doc_id': 3482, 'frequency': 1}, {'doc_id': 4810, 'frequency': 1}]
Term: 'aali' -> [{'doc_id': 587, 'frequency': 1}]
Term: 'aallayl' -> [{'doc_id': 239, 'frequency': 2}]
Term: 'aan' -> [{'doc_id': 3202, 'frequency': 2}]
Term: 'aasar' -> [{'doc_id': 3708, 'frequency': 1}]
Term: 'aasr' -> [{'doc_id': 4201, 'frequency': 1}]
Term: 'aassar' -> [{'doc_id': 5015, 'frequency': 1}]


In [56]:
search_term="egypt"
if search_term in inverted_index:
    print(f"Documents containing '{search_term}':")
    print(inverted_index[search_term])
else:
    print(f"Term '{search_term}' not found in index")

Documents containing 'egypt':
[{'doc_id': 2, 'frequency': 1}, {'doc_id': 3, 'frequency': 1}, {'doc_id': 5, 'frequency': 1}, {'doc_id': 15, 'frequency': 1}, {'doc_id': 17, 'frequency': 2}, {'doc_id': 40, 'frequency': 1}, {'doc_id': 48, 'frequency': 1}, {'doc_id': 60, 'frequency': 1}, {'doc_id': 105, 'frequency': 1}, {'doc_id': 107, 'frequency': 2}, {'doc_id': 109, 'frequency': 1}, {'doc_id': 116, 'frequency': 1}, {'doc_id': 185, 'frequency': 1}, {'doc_id': 203, 'frequency': 1}, {'doc_id': 224, 'frequency': 1}, {'doc_id': 227, 'frequency': 1}, {'doc_id': 233, 'frequency': 1}, {'doc_id': 256, 'frequency': 1}, {'doc_id': 264, 'frequency': 1}, {'doc_id': 282, 'frequency': 1}, {'doc_id': 289, 'frequency': 1}, {'doc_id': 295, 'frequency': 1}, {'doc_id': 303, 'frequency': 1}, {'doc_id': 360, 'frequency': 1}, {'doc_id': 398, 'frequency': 1}, {'doc_id': 402, 'frequency': 1}, {'doc_id': 417, 'frequency': 1}, {'doc_id': 423, 'frequency': 1}, {'doc_id': 428, 'frequency': 1}, {'doc_id': 462, 'freque

In [57]:
if search_term in inverted_index:
    results=inverted_index[search_term]
    print(f"Term:'{search_term}' appears in {len(results)} documents\n")
    print("First 5 entries:")
    for entry in results[:5]:
        print(f"  doc_id: {entry['doc_id']}, frequency: {entry['frequency']}")

Term:'egypt' appears in 344 documents

First 5 entries:
  doc_id: 2, frequency: 1
  doc_id: 3, frequency: 1
  doc_id: 5, frequency: 1
  doc_id: 15, frequency: 1
  doc_id: 17, frequency: 2


In [58]:
total_terms=len(inverted_index)
total_documents=len(df)
print(f"Total unique terms: {total_terms}")
print(f"Total documents: {total_documents}")

Total unique terms: 11168
Total documents: 5036


In [59]:
import re
import pandas as pd
terms_stats=[]
for x in index.getLexicon():
    term=x.getKey()
    if term.isdigit() or re.match(r'^\d+\w*$', term):
        continue
    stats=x.getValue()
    terms_stats.append({
        'term':term,
        'frequency':stats.getFrequency(),
        'doc_count':stats.getDocumentFrequency()
    })
terms_df=pd.DataFrame(terms_stats)
terms_df.head(20)

,term,frequency,doc_count
0,aa,4,4
1,aaan,1,1
2,aad,5,5
3,aal,5,4
4,aali,1,1
5,aallayl,2,1
6,aan,2,1
7,aasar,1,1
8,aasr,1,1
9,aassar,1,1


In [60]:
print("Sample Terms with Statistics:")
print(f"\n{'Term':<20} {'Total Frequency':<18} {'Document Count':<15}")
print("-" * 53)
for i in range(min(15, len(terms_df))):
    row = terms_df.iloc[i]
    print(f"{row['term']:<20} {row['frequency']:<18} {row['doc_count']:<15}")

Sample Terms with Statistics:

Term                 Total Frequency    Document Count 
-----------------------------------------------------
aa                   4                  4              
aaan                 1                  1              
aad                  5                  5              
aal                  5                  4              
aali                 1                  1              
aallayl              2                  1              
aan                  2                  1              
aasar                1                  1              
aasr                 1                  1              
aassar               1                  1              
ab                   10                 8              
aback                1                  1              
abadan               1                  1              
abal                 1                  1              
abalssa              1                  1              


In [65]:
print("\nOriginal vs Processed Text Example:")
print(f"\nOriginal: {df.iloc[0]['combined_text'][:100]}...")
print(f"\nProcessed: {df.iloc[0]['processed_text'][:100]}...")


Original vs Processed Text Example:

Original: The Venal Gentlemen Mahmoud, a simple employee of the Ministry of Health, loves lawyer "Amal", who t...

Processed: venal gentlemen mahmoud simpl employe ministri health love lawyer amal train offic teacher ali hamma...
